In [ ]:
from pathlib import Path
import sys

# Locate this package and load the shared v3 LE locations (config/paths.json,
# overridable with the V3LE_ROOT environment variable).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from paths import V3LE_DATA_DIR, V3LE_DIAG_DIR, V3LE_FIG_ROOT, fig_dir, require

In [4]:
import os, re, glob, json
from pathlib import Path
import numpy as np
import xarray as xr
import pandas as pd

In [5]:
class ArcticAAProcessor:
    """
    Arctic Amplification from region-mean files with anomalies and running trends.

    Files like:
      - ERA5.en00.region_means.197901-201812.nc   (or ERA5.region_means.*.nc)
      - v3.LR.historical.enXX.region_means.185001-205012.nc

    Variables expected: TS(region,time), region in ["global","Arctic"] (case-insensitive).

    This version strictly enforces *full-window* rolling trends for anchors:
      - trend_anchor: 'center' | 'start' | 'end'
      - Window length = round(trend_window / median_dt)
      - Only indices with the entire window inside the record are computed.
    """

    # --------------- init ---------------
    def __init__(
        self,
        *,
        data_dir,
        var="TS",
        pclimo=(1981, 2010),
        ptargt=(1979, 2018),
        trend_window=30,                      # years
        seasons=("ANN", "DJF", "MAM", "JJA", "SON"),
        aa_gate=2.0,                          # threshold for "AA>2" fraction
        region_loc="Arctic",
        region_glb="global",
        verbose=True,
        center_only=True,                     # keep only indices with full windows
        trend_anchor="center",                # 'center' | 'start' | 'end'
        valid_policy="any",                   # 'all' or 'any' across ensemble
        denom_eps=1e-3,                       # K/decade guard for denominator
        compress_level=2,                     # NetCDF deflate level
    ):
        self.data_dir = data_dir
        self.var = var
        self.pclimo = tuple(pclimo)
        self.ptargt = tuple(ptargt)
        self.trend_window = int(trend_window)
        self.seasons = tuple(s.upper() for s in seasons)
        self.aa_gate = float(aa_gate)
        self.region_loc=region_loc
        self.region_glb=region_glb
        self.verbose = verbose
        self.center_only = bool(center_only)
        self.trend_anchor = str(trend_anchor).lower()
        assert self.trend_anchor in ("center", "start", "end")
        self.valid_policy = str(valid_policy).lower()
        assert self.valid_policy in ("all", "any")
        self.denom_eps = float(denom_eps)
        self.compress_level = int(compress_level)

    # --------------- helpers ---------------
    @staticmethod
    def _ens_index(path):
        m = re.search(r"\.en(\d+)\.", os.path.basename(path))
        return int(m.group(1)) if m else -1

    @staticmethod
    def _to_decimal_years(time_coord):
        """Robust conversion of CF time to decimal years (supports cftime calendars)."""
        try:
            ti = pd.to_datetime(time_coord.values)
            return ti.year + (ti.dayofyear - 0.5) / 365.25
        except Exception:
            return np.array(
                [t.year + (t.timetuple().tm_yday - 0.5) / 365.25 for t in time_coord.values],
                dtype=float,
            )

    @staticmethod
    def _seasonal_mean_monthly(ds, season):
        """Seasonal/annual mean from monthly data; DJF via QS-DEC, labeled to Jan's year."""
        if season == "ANN":
            return ds.resample(time="YS").mean(keep_attrs=True)
        qs = ds.resample(time="QS-DEC").mean(keep_attrs=True)
        pick = {"DJF": 0, "MAM": 1, "JJA": 2, "SON": 3}[season]
        sel = qs.isel(time=(np.arange(qs.sizes["time"]) % 4 == pick))
        if season == "DJF":
            t = sel["time"].values
            try:
                jan_year = (pd.DatetimeIndex(t) + pd.offsets.MonthBegin(1)).year
            except Exception:
                jan_year = np.array([getattr(tt, "year", np.nan) for tt in t], dtype=float)
            sel = sel.assign_coords(time=("time", pd.to_datetime(jan_year, format="%Y")))
        return sel

    @staticmethod
    def _window_counts(years, win_years):
        """Return (win_n, dt) where win_n = round(win_years / median_dt)."""
        years = np.asarray(years, float)
        if years.size < 2:
            return 1, 1.0
        dt = np.nanmedian(np.diff(years))
        win_n = max(3, int(round(win_years / dt)))
        return win_n, dt

    @staticmethod
    def _running_linear_trend_full(y, x, win_years, anchor="center"):
        """
        Rolling OLS slope (K/yr) over a *full* window of length win_n samples.
        Only computes at indices where the entire window is within bounds.
        Requires all win_n samples finite (strict).
        """
        y = np.asarray(y, float)
        x = np.asarray(x, float)
        n = y.size
        out = np.full(n, np.nan)
        if n < 3:
            return out

        win_n, _ = ArcticAAProcessor._window_counts(x, win_years)
        anchor = str(anchor).lower()
        assert anchor in ("center", "start", "end")

        if anchor == "center":
            half = win_n // 2
            # even win_n (e.g., 30) -> i0=i-half, i1=i+half  (size = win_n)
            for i in range(half, n - (win_n - half) + 1):
                i0 = i - half
                i1 = i + half if (win_n % 2 == 0) else i + half + 1
                xx = x[i0:i1]
                yy = y[i0:i1]
                msk = np.isfinite(xx) & np.isfinite(yy)
                if msk.sum() != win_n:
                    continue
                X = np.vstack([xx, np.ones(win_n)]).T
                beta, *_ = np.linalg.lstsq(X, yy, rcond=None)
                out[i] = beta[0]
            return out

        if anchor == "start":
            last = n - win_n
            if last < 0:
                return out
            for i in range(0, last + 1):
                i0, i1 = i, i + win_n
                xx = x[i0:i1]; yy = y[i0:i1]
                msk = np.isfinite(xx) & np.isfinite(yy)
                if msk.sum() != win_n:
                    continue
                X = np.vstack([xx, np.ones(win_n)]).T
                beta, *_ = np.linalg.lstsq(X, yy, rcond=None)
                out[i] = beta[0]
            return out

        # anchor == 'end'
        first = win_n - 1
        if first >= n:
            return out
        for i in range(first, n):
            i0, i1 = i - win_n + 1, i + 1
            xx = x[i0:i1]; yy = y[i0:i1]
            msk = np.isfinite(xx) & np.isfinite(yy)
            if msk.sum() != win_n:
                continue
            X = np.vstack([xx, np.ones(win_n)]).T
            beta, *_ = np.linalg.lstsq(X, yy, rcond=None)
            out[i] = beta[0]
        return out

    def _label_years_for_anchor_full(self, years, win_years, anchor):
        """Return (label_year, left_year, right_year) only for full-window indices."""
        years = np.asarray(years, float)
        n = years.size
        lab = np.full(n, np.nan)
        L   = np.full(n, np.nan)
        R   = np.full(n, np.nan)
        if n < 1:
            return lab, L, R

        win_n, _ = self._window_counts(years, win_years)
        anchor = str(anchor).lower()

        if anchor == "center":
            half = win_n // 2
            for i in range(half, n - (win_n - half) + (0 if win_n % 2 == 0 else 1)):
                i0 = i - half
                i1 = i + half if (win_n % 2 == 0) else i + half + 1
                L[i], R[i] = years[i0], years[i1 - 1]
                lab[i] = 0.5 * (L[i] + R[i])
            return lab, L, R

        if anchor == "start":
            last = n - win_n
            if last >= 0:
                for i in range(0, last + 1):
                    L[i], R[i] = years[i], years[i + win_n - 1]
                    lab[i] = L[i]
            return lab, L, R

        # end
        first = win_n - 1
        if first < n:
            for i in range(first, n):
                L[i], R[i] = years[i - win_n + 1], years[i]
                lab[i] = R[i]
        return lab, L, R

    # --------------- IO / discovery ---------------
    def discover_paths(self, group, period, nens):
        y0m, y1m = f"{period[0]}01", f"{period[1]}12"
        if nens == 1:
            p1 = os.path.join(self.data_dir, f"{group}.en00.region_means.{y0m}-{y1m}.nc")
            p2 = os.path.join(self.data_dir, f"{group}.region_means.{y0m}-{y1m}.nc")
            paths = [p for p in (p1, p2) if os.path.exists(p)]
        else:
            pat = os.path.join(self.data_dir, f"{group}.en??.region_means.{y0m}-{y1m}.nc")
            paths = sorted(glob.glob(pat), key=self._ens_index)
        if self.verbose:
            print(f"[discover] group={group} nens={nens} -> {len(paths)} file(s)")
        if not paths:
            raise FileNotFoundError(f"No files found for group={group}, period={period}, nens={nens}")
        return paths

    def _open_one(self, path):
        ds = xr.open_dataset(path, decode_times=True, use_cftime=True)
        if "region" not in ds.dims:
            raise ValueError(f"'region' dimension missing in {path}")
        regs = [str(r) for r in ds["region"].values]

        def pick(name):
            return next((r for r in regs if r.lower() == name.lower()), None)

        g = pick(self.region_glb); 
        a = pick(self.region_loc)
        if g is None or a is None:
            raise ValueError(f"Expected regions [{self.region_glb},{self.region_loc}] missing in {path}")

        return ds[[self.var]].sel(region=[g, a])

    def _concat_ens(self, paths):
        dsets = []
        for p in paths:
            ds = self._open_one(p)
            ens = f"en{str(self._ens_index(p)).zfill(2)}"
            dsets.append(ds.expand_dims(ens=[ens]))
        if len(dsets) == 1 and "ens" not in dsets[0].dims:
            dsets[0] = dsets[0].expand_dims(ens=["en00"])
        return xr.concat(dsets, dim="ens")

    # --------------- Core steps ---------------
    def _monthly_anomaly(self, da):
        """Anomaly by removing monthly climatology over fixed PCLIMO."""
        try:
            year = da["time"].dt.year
        except Exception:
            year = xr.DataArray(np.floor(self._to_decimal_years(da["time"])).astype(int), dims=("time",))
        base = da.sel(time=(year >= self.pclimo[0]) & (year <= self.pclimo[1]))
        clim = base.groupby("time.month").mean("time")
        return da.groupby("time.month") - clim

    def _season_reduce(self, ds, season):
        return self._seasonal_mean_monthly(ds, season)

    def compute_for_season(self, paths, season):
        """
        Returns:
          ds_out with coords time (trimmed to full-window indices if center_only=True),
          time_label_year, time_window_start_year, time_window_end_year,
          and variables:
            - aa, beta_arc, beta_glb (K/decade for betas; AA = arc/glb)
            - aa_mean/std, beta_*_mean/std
          stats: ensemble summary over kept indices within PTARGT.
        """
        ds = self._concat_ens(paths)                      # (ens, region, time)
        DT = self._monthly_anomaly(ds[self.var])          # monthly anomalies
        DTs = self._season_reduce(DT, season)             # seasonal/annual means

        # regions (order enforced earlier): [global, Arctic]
        glb = DTs.sel(region=DTs.region.values[0])
        arc = DTs.sel(region=DTs.region.values[1])

        # restrict to PTARGT via decimal years
        yrs_all = self._to_decimal_years(DTs["time"])
        msk = (yrs_all >= self.ptargt[0]) & (yrs_all <= self.ptargt[1])
        glb = glb.sel(time=msk)
        arc = arc.sel(time=msk)
        yrs = yrs_all[msk]

        # per-member rolling trends (full-window only) -> K/decade
        bet_glb_arr, bet_arc_arr = [], []
        for i in range(glb.sizes["ens"]):
            yg = glb.isel(ens=i).values
            ya = arc.isel(ens=i).values
            bg = self._running_linear_trend_full(yg, yrs, self.trend_window, anchor=self.trend_anchor) * 10.0
            ba = self._running_linear_trend_full(ya, yrs, self.trend_window, anchor=self.trend_anchor) * 10.0
            bet_glb_arr.append(bg)
            bet_arc_arr.append(ba)

        bet_glb = xr.DataArray(
            np.vstack(bet_glb_arr), dims=("ens", "time"), coords={"ens": glb["ens"], "time": glb["time"]}
        )
        bet_arc = xr.DataArray(
            np.vstack(bet_arc_arr), dims=("ens", "time"), coords={"ens": arc["ens"], "time": arc["time"]}
        )

        # Full-window mask is simply where center (or start/end) indices were computed (finite across ensemble)
        if self.valid_policy == "all":
            valid = np.isfinite(bet_glb).all(dim="ens") & np.isfinite(bet_arc).all(dim="ens")
        else:
            valid = np.isfinite(bet_glb).any(dim="ens") & np.isfinite(bet_arc).any(dim="ens")

        # Label years + window edges based on full-window indices
        lab_year_all, L_year_all, R_year_all = self._label_years_for_anchor_full(yrs, self.trend_window, self.trend_anchor)

        if self.center_only:
            bet_glb = bet_glb.sel(time=valid)
            bet_arc = bet_arc.sel(time=valid)
            idx = np.where(valid.values)[0]
            lab_year = lab_year_all[idx]
            L_year   = L_year_all[idx]
            R_year   = R_year_all[idx]
        else:
            lab_year = lab_year_all
            L_year   = L_year_all
            R_year   = R_year_all

        # AA ratio with stable denominator (K/decade)
        denom = xr.where(np.abs(bet_glb) < self.denom_eps, np.nan, bet_glb)
        aa = bet_arc / denom

        # coords
        time_label_year = xr.DataArray(
            lab_year, dims=("time",), coords={"time": bet_glb["time"]}, name="time_label_year"
        )
        time_window_start_year = xr.DataArray(
            L_year, dims=("time",), coords={"time": bet_glb["time"]}, name="time_window_start_year"
        )
        time_window_end_year = xr.DataArray(
            R_year, dims=("time",), coords={"time": bet_glb["time"]}, name="time_window_end_year"
        )
        
        # Full uniform PTARGT axis (integer years)
        yrs_full = np.arange(self.ptargt[0], self.ptargt[1] + 1, dtype=int)
        coord_full = xr.DataArray(yrs_full, dims=("time_year",), name="time_year")

        # Round only valid labels to integer years for indexing
        lab_valid = np.isfinite(lab_year)
        yrs_label_int = np.round(lab_year[lab_valid]).astype(int)

        # Helper to reindex a 1D array using the integer labels
        def _reindex_1d(arr):
            arr = xr.DataArray(arr[lab_valid], dims=("time",))
            arr = arr.assign_coords(time_year=("time", yrs_label_int))
            return arr.swap_dims({"time": "time_year"}).reindex(time_year=yrs_full)

        # Reindex members along time_year
        aa = aa.sel(time=lab_valid)
        bet_glb = bet_glb.sel(time=lab_valid)
        bet_arc = bet_arc.sel(time=lab_valid)
                
        aa = aa.assign_coords(time_year=("time", yrs_label_int)).swap_dims({"time": "time_year"}).reindex(time_year=yrs_full)
        bet_glb = bet_glb.assign_coords(time_year=("time", yrs_label_int)).swap_dims({"time": "time_year"}).reindex(time_year=yrs_full)
        bet_arc = bet_arc.assign_coords(time_year=("time", yrs_label_int)).swap_dims({"time": "time_year"}).reindex(time_year=yrs_full)

        # Reindex label/edge coordinates
        lab_year_full = _reindex_1d(lab_year)
        L_year_full   = _reindex_1d(L_year)
        R_year_full   = _reindex_1d(R_year)
        
        # Drop stray 'time' coordinate (non-numeric) to avoid dtype inference issues
        for v in (aa, bet_glb, bet_arc):
            if "time" in v.coords:
                v = v.drop_vars("time")
                
        # dataset (now time dimension is 'time_year' for everything)
        ds_out = xr.Dataset(
            dict(
                aa=aa,
                beta_arc=bet_arc,
                beta_glb=bet_glb,
                aa_mean=aa.mean("ens"),
                aa_std=aa.std("ens"),
                beta_arc_mean=bet_arc.mean("ens"),
                beta_arc_std=bet_arc.std("ens"),
                beta_glb_mean=bet_glb.mean("ens"),
                beta_glb_std=bet_glb.std("ens"),
                time_label_year=lab_year_full,
                time_window_start_year=L_year_full,
                time_window_end_year=R_year_full,
            ),
            coords=dict(time_year=coord_full),
        )
        if self.trend_anchor == "center":
            ds_out["time_center_year"] = ds_out["time_label_year"]

        ds_out.attrs.update(
            var=self.var,
            pclimo=f"{self.pclimo[0]}-{self.pclimo[1]}",
            ptargt=f"{self.ptargt[0]}-{self.ptargt[1]}",
            trend_window_years=self.trend_window,
            trend_anchor=self.trend_anchor,
            season=season,
            center_only=str(self.center_only),
            valid_policy=self.valid_policy,
            n_time_fixed=int(yrs_full.size),
            note="AA padded to full PTARGT window with NaNs for missing years (full-window indices only).",
        )

        # ---- summary stats over kept (non-NaN) time positions ----
        # valid where any ensemble has finite AA (or use 'all' if you prefer)
        valid_time = np.isfinite(aa).any(dim="ens")
        aa_bar   = aa.sel(time_year=valid_time).mean("time_year")     # per-member scalar
        bar_arc  = bet_arc.sel(time_year=valid_time).mean("time_year")
        bar_glb  = bet_glb.sel(time_year=valid_time).mean("time_year")
        frac_gt2 = (aa.sel(time_year=valid_time) > self.aa_gate).mean("time_year")

        def pct(arr, q): return float(np.nanpercentile(arr, q))

        stats = {
            "season": season,
            "trend_window_years": self.trend_window,
            "trend_anchor": self.trend_anchor,
            "ptargt": self.ptargt,
            "pclimo": self.pclimo,
            "AA_gate": self.aa_gate,
            "n_ens": int(aa.sizes["ens"]),
            "n_time": int(valid_time.sum().item()),  # number of non-NaN years (full-window count)
            "AA_bar_mean": float(aa_bar.mean("ens").values),
            "AA_bar_std":  float(aa_bar.std("ens").values),
            "AA_bar_p05":  pct(aa_bar.values, 5),
            "AA_bar_p50":  pct(aa_bar.values, 50),
            "AA_bar_p95":  pct(aa_bar.values, 95),
            "beta_arc_bar_mean": float(bar_arc.mean("ens").values),
            "beta_glb_bar_mean": float(bar_glb.mean("ens").values),
            "frac_time_AA_gt_gate_mean": float(frac_gt2.mean("ens").values),
        }
        return ds_out, stats

    # --------------- Export helpers ---------------
    def save_netcdf(self, ds, path):
        Path(os.path.dirname(path)).mkdir(parents=True, exist_ok=True)

        # --- Drop non-numeric "time" coordinate if it exists ---
        if "time" in ds.coords:
            if not np.issubdtype(ds["time"].dtype, np.number):
                if self.verbose:
                    print(f"[WARN] Dropping non-numeric coord 'time' before saving → {path}")
                ds = ds.drop_vars("time")

        # --- Encoding setup ---
        enc = {
            v: {"zlib": True, "complevel": self.compress_level, "dtype": "float32"}
            for v in ds.data_vars
        }
        for c in ("time_label_year", "time_window_start_year", "time_window_end_year", "time_center_year"):
            if c in ds:
                enc[c] = {"zlib": True, "complevel": self.compress_level, "dtype": "float32"}

        ds.load().to_netcdf(path, encoding=enc)

    def save_stats_json(self, stats, path):
        Path(os.path.dirname(path)).mkdir(parents=True, exist_ok=True)
        with open(path, "w") as f:
            json.dump(stats, f, indent=2)

    # --------------- Convenience ---------------
    def run_and_save(self, *, group, period, nens, out_dir, tag_prefix="AA"):
        paths = self.discover_paths(group, period, nens)
        results = {}
        for seas in self.seasons:
            ds_out, stats = self.compute_for_season(paths, season=seas)
            tag = (
                f"{tag_prefix}_{self.trend_anchor}_ts_{group}_{self.var}_{seas}_"
                f"{self.ptargt[0]}-{self.ptargt[1]}_{self.trend_window}yr_{self.trend_anchor}"
            )
            nc_out = os.path.join(out_dir, f"{tag}.nc")
            js_out = os.path.join(out_dir, f"{tag}.json")
            self.save_netcdf(ds_out, nc_out)
            self.save_stats_json(stats, js_out)
            if self.verbose:
                print(f"[OK] wrote {nc_out}")
                print(f"[OK] wrote {js_out}")
            results[seas] = {"nc": nc_out, "json": js_out, "stats": stats}
        return results

    def run_and_save_anchors(self, *, group, period, nens, out_dir, tag_prefix="AA", anchors=("center","start","end")):
        out = {}
        for anch in anchors:
            prev = self.trend_anchor
            self.trend_anchor = anch
            try:
                res = self.run_and_save(
                    group=group, 
                    period=period, 
                    nens=nens, 
                    out_dir=out_dir, 
                    tag_prefix=f"{tag_prefix}"
                )
            finally:
                self.trend_anchor = prev
            out[anch] = res
        return out


In [6]:
if __name__ == "__main__":
    DATA_DIR  = f"{V3LE_DATA_DIR}/regmn_driftcorr"
    OUT_DIR   = str(V3LE_DIAG_DIR)
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

    # ---- Pair experiments consistently (order matters) ----
    EXPS    = ["ERA5",        "E3SMv3-LE"]
    GROUP   = ["ERA5",        "v3.LR.historical"]
    PERIOD  = [(1979, 2018),  (1850, 2050)]   # ERA5 first, E3SM second
    NENS    = [1,             25]
    
    region_dict = {
        "arctic": {"name": "Arctic2", "lat_range": (66.5, 90.0),  "lon_range": (-180, 180)},
        "global": {"name": "global", "lat_range": (-90.0, 90.0), "lon_range": (-180, 180)},
    }
    
    # ---- AA config ----
    PTARGT  = (1979, 2050)    # analysis window for AA (ERA5 will naturally truncate to 1979–2018)
    PCLIMO  = (1981, 2010)
    VAR     = "TS"
    SEASONS = ["ANN", "DJF", "MAM", "JJA", "SON"]

    trend_window = 30          # running window in years
    aa_gate = 2.0
    verbose = True

    # ---- Instantiate the processor (region_means workflow: no grids needed) ----
    engine = ArcticAAProcessor(
        data_dir=DATA_DIR,
        var=VAR,
        pclimo=PCLIMO,
        ptargt=PTARGT,
        trend_window=trend_window,
        seasons=SEASONS,
        aa_gate=aa_gate,
        region_loc=region_dict["arctic"]["name"],
        region_glb=region_dict["global"]["name"],
        verbose=verbose,
        # Optional robustness knobs (uncomment to enable):
        # valid_policy="all",      # require all members valid at a time index
        # denom_eps=1e-3,          # guard tiny global slopes (K/decade)
        # compress_level=2,        # NetCDF deflate level
        # center_only=True,        # keep only valid positions
    )

    manifest = {}
    for exp, grp, prd, nens in zip(EXPS, GROUP, PERIOD, NENS):
        print(f"\n[RUN] {exp} | group={grp} | period={prd[0]}-{prd[1]} | nens={nens}")

        # Write all three anchoring variants in one pass
        results_by_anchor = engine.run_and_save_anchors(
            group=grp,
            period=prd,
            nens=nens,
            out_dir=OUT_DIR,
            tag_prefix=f"AA2_driftcorr_{exp}",
            anchors=("center", "start", "end"),
        )
        manifest[exp] = results_by_anchor

        # Compact summary printout
        for anch, results in results_by_anchor.items():
            for seas, rec in results.items():
                s = rec["stats"]
                print(
                    f"  [{anch}] {seas}: "
                    f"{os.path.basename(rec['nc'])}, {os.path.basename(rec['json'])} | "
                    f"n_ens={s['n_ens']}, n_time={s['n_time']}, "
                    f"AĀ={s['AA_bar_mean']:.2f}±{s['AA_bar_std']:.2f}, "
                    f"P(AA>{s['AA_gate']})={s['frac_time_AA_gt_gate_mean']:.2f}"
                )

    # Save a run manifest for quick lookup
    with open(Path(OUT_DIR, "AA2_driftcorr_manifest.anchors.json"), "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"\n[OK] Wrote manifest: {Path(OUT_DIR, 'AA2_driftcorr_manifest.anchors.json')}")



[RUN] ERA5 | group=ERA5 | period=1979-2018 | nens=1
[discover] group=ERA5 nens=1 -> 1 file(s)
[WARN] Dropping non-numeric coord 'time' before saving → /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_ERA5_center_ts_ERA5_TS_ANN_1979-2050_30yr_center.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_ERA5_center_ts_ERA5_TS_ANN_1979-2050_30yr_center.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_ERA5_center_ts_ERA5_TS_ANN_1979-2050_30yr_center.json
[WARN] Dropping non-numeric coord 'time' before saving → /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_ERA5_center_ts_ERA5_TS_DJF_1979-2050_30yr_center.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_ERA5_center_ts_ERA5_TS_DJF_1979-2050_30yr_center.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acm

[WARN] Dropping non-numeric coord 'time' before saving → /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_E3SMv3-LE_center_ts_v3.LR.historical_TS_ANN_1979-2050_30yr_center.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_E3SMv3-LE_center_ts_v3.LR.historical_TS_ANN_1979-2050_30yr_center.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_E3SMv3-LE_center_ts_v3.LR.historical_TS_ANN_1979-2050_30yr_center.json
[WARN] Dropping non-numeric coord 'time' before saving → /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_E3SMv3-LE_center_ts_v3.LR.historical_TS_DJF_1979-2050_30yr_center.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/e3sm_project/v3_le_paper/figure_data/AA2_driftcorr_E3SMv3-LE_center_ts_v3.LR.historical_TS_DJF_1979-2050_30yr_center.nc
[OK] wrote /lcrc/group/e3sm/ac.szhang/acme_scratch/